# 03 Modeling v2 — Polymarket Prediction Markets

Binary classification: predict whether a market will reach high liquidity (volume >= 75th percentile).

This notebook:
- Loads the enriched feature matrix `data/features_v2.parquet`.
- Compares several classifiers with stratified cross-validation.
- Tunes the top two models with randomized search.
- Evaluates final models with ROC-AUC, PR-AUC, classification reports and confusion matrices.
- Compares feature importances and explains the best model with SHAP.

In [ ]:
import json
import logging
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import xgboost as xgb
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# Logging instead of print
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Plotting defaults
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

# Paths
ROOT = Path('/home/diego/Proyectos-DS/Polymarket_Prediction_Markets')
DATA_PATH = ROOT / 'data' / 'features_v2.parquet'

In [ ]:
logger.info('Loading feature matrix...')
df = pd.read_parquet(DATA_PATH)

logger.info('Dataset shape: %s', df.shape)
logger.info('Columns (%d): %s', len(df.columns), df.columns.tolist())
logger.info('Class balance:\n%s', df['high_liquidity'].value_counts(normalize=True).round(4).to_string())

## 1. Train / test split

We use a stratified 80/20 split to preserve the ~24% positive-class ratio.

In [ ]:
TARGET = 'high_liquidity'
X = df.drop(TARGET, axis=1)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

logger.info('Train shape: %s, Test shape: %s', X_train.shape, X_test.shape)
logger.info('Train positive rate: %.4f', y_train.mean())
logger.info('Test positive rate: %.4f', y_test.mean())

## 2. Baseline models with stratified CV

We compare five classifiers. Tree-based models receive a `scale_pos_weight`; linear and forest models use `class_weight='balanced'`.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
logger.info('Scale pos weight: %.4f', scale_pos_weight)

models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1))
    ]),
    'RandomForest': RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200,
        max_depth=4,
        random_state=42
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        use_label_encoder=False,
        eval_metric='logloss'
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    logger.info('Cross-validating %s...', name)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results.append({
        'model': name,
        'roc_auc_mean': scores.mean(),
        'roc_auc_std': scores.std()
    })
    logger.info('%s CV ROC-AUC: %.4f (+/- %.4f)', name, scores.mean(), scores.std())

cv_df = pd.DataFrame(cv_results).sort_values('roc_auc_mean', ascending=False)
cv_df

## 3. Hyperparameter tuning for top models

We run a randomized search on the two best CV performers to keep runtime reasonable.

In [ ]:
top_models = cv_df.head(2)['model'].tolist()
logger.info('Tuning top models: %s', top_models)

param_grids = {
    'LightGBM': {
        'n_estimators': [200, 400, 600],
        'learning_rate': [0.03, 0.05, 0.1],
        'num_leaves': [15, 31, 63],
        'max_depth': [-1, 6, 10],
        'min_child_samples': [10, 20, 50]
    },
    'XGBoost': {
        'n_estimators': [200, 400, 600],
        'learning_rate': [0.03, 0.05, 0.1],
        'max_depth': [4, 6, 8],
        'min_child_weight': [1, 3, 5],
        'subsample': [0.8, 1.0]
    },
    'RandomForest': {
        'n_estimators': [200, 400],
        'max_depth': [10, 15, 20, None],
        'min_samples_leaf': [1, 2, 5]
    },
    'GradientBoosting': {
        'n_estimators': [200, 400],
        'learning_rate': [0.03, 0.05, 0.1],
        'max_depth': [3, 4, 5]
    },
    'LogisticRegression': {
        'clf__C': [0.01, 0.1, 1, 10],
        'clf__penalty': ['l1', 'l2']
    }
}

tuned_models = {}

for name in top_models:
    logger.info('Tuning %s...', name)
    model = models[name]
    grid = param_grids[name]
    search = RandomizedSearchCV(
        model,
        grid,
        n_iter=15 if name != 'LogisticRegression' else 8,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    logger.info('%s best CV ROC-AUC: %.4f', name, search.best_score_)
    logger.info('%s best params: %s', name, search.best_params_)

## 4. Final evaluation on hold-out test set

For each tuned model we report ROC-AUC, PR-AUC and a classification report.

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    """Compute and log key binary-classification metrics."""
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    logger.info('--- %s test evaluation ---', name)
    logger.info('ROC-AUC: %.4f', roc_auc)
    logger.info('PR-AUC:  %.4f', pr_auc)
    logger.info('Classification report:\n%s', classification_report(y_test, y_pred, digits=4))
    return {'model': name, 'roc_auc': roc_auc, 'pr_auc': pr_auc}

test_results = []
for name, model in tuned_models.items():
    test_results.append(evaluate_model(name, model, X_test, y_test))

test_df = pd.DataFrame(test_results).sort_values('roc_auc', ascending=False)
test_df

## 5. ROC and Precision-Recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, model in tuned_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name=name)
    PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], name=name)

axes[0].set_title('ROC curves')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_title('Precision-Recall curves')
baseline = y_test.mean()
axes[1].axhline(baseline, color='k', linestyle='--', lw=1, label=f'Baseline ({baseline:.3f})')
axes[1].legend(loc='best')
plt.tight_layout()
plt.show()

## 6. Confusion matrices

In [ ]:
n_models = len(tuned_models)
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, tuned_models.items()):
    y_pred = model.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues')
    ax.set_title(f'{name} confusion matrix')

plt.tight_layout()
plt.show()

## 7. Feature importance comparison

In [ ]:
importance_dfs = []

for name, model in tuned_models.items():
    if name == 'LogisticRegression':
        importances = np.abs(model.named_steps['clf'].coef_[0])
    elif hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    else:
        continue
    imp_df = pd.DataFrame({
        'feature': X.columns,
        'importance': importances,
        'model': name
    }).sort_values('importance', ascending=False).head(15)
    importance_dfs.append(imp_df)

imp_all = pd.concat(importance_dfs)

g = sns.FacetGrid(imp_all, col='model', col_wrap=2, sharey=False, height=5, aspect=1.2)
g.map_dataframe(sns.barplot, x='importance', y='feature', palette='viridis')
g.set_titles(col_template='{col_name}')
g.tight_layout()
plt.show()

## 8. SHAP explainability

We explain the best-performing tree-based model with SHAP TreeExplainer on a stratified sample of 1,000 test observations.

In [ ]:
best_tree_name = [m for m in tuned_models if m != 'LogisticRegression'][0]
best_tree_model = tuned_models[best_tree_name]
logger.info('Explaining model: %s', best_tree_name)

# Stratified sample for SHAP to keep both classes
shap_sample = (
    X_test.groupby(y_test, group_keys=False)
    .apply(lambda x: x.sample(min(500, len(x)), random_state=42))
)
shap_labels = y_test.loc[shap_sample.index]

explainer = shap.TreeExplainer(best_tree_model)
shap_values = explainer.shap_values(shap_sample)

shap.summary_plot(shap_values, shap_sample, plot_type='bar', show=False, max_display=20)
plt.title(f'SHAP feature importance — {best_tree_name}')
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values, shap_sample, show=False, max_display=20)
plt.title(f'SHAP beeswarm — {best_tree_name}')
plt.tight_layout()
plt.show()

## 9. Save best model artifacts

Persist the best model and a small report for downstream use.

In [ ]:
import joblib

best_name = test_df.iloc[0]['model']
best_model = tuned_models[best_name]

model_path = ROOT / 'data' / 'best_model_v2.joblib'
joblib.dump(best_model, model_path)
logger.info('Saved best model to %s', model_path)

report = {
    'best_model': best_name,
    'test_roc_auc': float(test_df.iloc[0]['roc_auc']),
    'test_pr_auc': float(test_df.iloc[0]['pr_auc']),
    'cv_results': cv_df.to_dict('records'),
    'test_results': test_df.to_dict('records'),
}
report_path = ROOT / 'data' / 'modeling_report_v2.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
logger.info('Saved modeling report to %s', report_path)

## Conclusions

- Several gradient-boosted and tree-based classifiers were benchmarked with 5-fold stratified CV.
- The top two models were tuned with `RandomizedSearchCV` and evaluated on a hold-out test set.
- ROC-AUC and PR-AUC are the primary metrics; PR-AUC is especially informative given the ~24% positive-class imbalance.
- SHAP analysis highlights which event-level and structural features drive the best model's predictions.
- The best model and a JSON report are saved under `data/` for reproducibility and downstream use.